<a href="https://colab.research.google.com/github/ashikjoel/-Context-Aware-Neural-Recommendation-Engine/blob/ashik_joel/Context_Aware_Neural_Recommendation_Engine(week_1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Install PySpark  (used to process large dataset)

In [2]:
!pip install -q pyspark

#Import PySpark  (let us to work with spark)

In [3]:
from pyspark.sql import SparkSession

#Create the Spark Session

In [4]:
spark = (
    SparkSession.builder
    .appName("HM-Recommendation-System")
    .getOrCreate()
)

In [5]:
print("Spark Version:", spark.version)

Spark Version: 4.0.3


#Downloading the H&M Dataset from kaggle

In [ ]:
from google.colab import userdata
api_key=userdata.get('kaggle')

In [7]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [9]:
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations

100% 28.7G/28.7G [04:46<00:00, 107MB/s]



In [10]:
!unzip -q h-and-m-personalized-fashion-recommendations.zip -d hm_dataset

In [11]:
import os

print(os.listdir("hm_dataset"))

['images', 'articles.csv', 'sample_submission.csv', 'transactions_train.csv', 'customers.csv']


#Loading the Dataset into PySpark

#Define the Dataset Path

In [12]:
DATA_PATH = "/content/hm_dataset"

#Load customers.csv

In [13]:
customers_df = spark.read.csv(
    f"{DATA_PATH}/customers.csv",
    header=True,
    inferSchema=True
)

#Load articles.csv

In [14]:
articles_df = spark.read.csv(
    f"{DATA_PATH}/articles.csv",
    header=True,
    inferSchema=True
)

#Load transactions_train.csv

In [15]:
transactions_df = spark.read.csv(
    f"{DATA_PATH}/transactions_train.csv",
    header=True,
    inferSchema=True
)

#Inspect the Data

In [16]:
customers_df.show(5)

+--------------------+----+------+------------------+----------------------+---+--------------------+
|         customer_id|  FN|Active|club_member_status|fashion_news_frequency|age|         postal_code|
+--------------------+----+------+------------------+----------------------+---+--------------------+
|00000dbacae5abe5e...|NULL|  NULL|            ACTIVE|                  NONE| 49|52043ee2162cf5aa7...|
|0000423b00ade9141...|NULL|  NULL|            ACTIVE|                  NONE| 25|2973abc54daa8a5f8...|
|000058a12d5b43e67...|NULL|  NULL|            ACTIVE|                  NONE| 24|64f17e6a330a85798...|
|00005ca1c9ed5f514...|NULL|  NULL|            ACTIVE|                  NONE| 54|5d36574f52495e81f...|
|00006413d8573cd20...| 1.0|   1.0|            ACTIVE|             Regularly| 52|25fa5ddee9aac01b3...|
+--------------------+----+------+------------------+----------------------+---+--------------------+
only showing top 5 rows


In [17]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- FN: double (nullable = true)
 |-- Active: double (nullable = true)
 |-- club_member_status: string (nullable = true)
 |-- fashion_news_frequency: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- postal_code: string (nullable = true)



In [18]:
customers_df.count()

1371980

In [19]:
articles_df.show(5)

+----------+------------+-----------------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------------+--------------+----------------+----------+--------------------+----------------+------------------+--------------------+
|article_id|product_code|        prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|department_name|index_code|      index_name|index_group_no|index_group_name|section_no|        section_name|garment_group_no|garment_group_name|         detail_desc|
+----------+------------+-----------------+-------------

In [20]:
articles_df.printSchema()

root
 |-- article_id: integer (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- index_code: string (nullable = true)
 |-- index_name: string (nullable = true)
 |-- index_group_no: integer (nullable = true)
 |-- index_group_name: string (nullable = true)

In [21]:
articles_df.count()

105542

In [22]:
transactions_df.show(5)

+----------+--------------------+----------+--------------------+----------------+
|     t_dat|         customer_id|article_id|               price|sales_channel_id|
+----------+--------------------+----------+--------------------+----------------+
|2018-09-20|000058a12d5b43e67...| 663713001|0.050830508474576264|               2|
|2018-09-20|000058a12d5b43e67...| 541518023| 0.03049152542372881|               2|
|2018-09-20|00007d2de826758b6...| 505221004| 0.01523728813559322|               2|
|2018-09-20|00007d2de826758b6...| 685687003|0.016932203389830508|               2|
|2018-09-20|00007d2de826758b6...| 685687004|0.016932203389830508|               2|
+----------+--------------------+----------+--------------------+----------------+
only showing top 5 rows


In [23]:
transactions_df.printSchema()

root
 |-- t_dat: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: integer (nullable = true)



In [24]:
transactions_df.count()

31788324

#Missing Values in Customers

In [25]:
from pyspark.sql.functions import col, count, when

customers_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in customers_df.columns
]).show()

+-----------+------+------+------------------+----------------------+-----+-----------+
|customer_id|    FN|Active|club_member_status|fashion_news_frequency|  age|postal_code|
+-----------+------+------+------------------+----------------------+-----+-----------+
|          0|895050|907576|              6062|                 16009|15861|          0|
+-----------+------+------+------------------+----------------------+-----+-----------+



#Missing Values in Articles

In [26]:
articles_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in articles_df.columns
]).show()

+----------+------------+---------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------+--------------+----------------+----------+------------+----------------+------------------+-----------+
|article_id|product_code|prod_name|product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|department_name|index_code|index_name|index_group_no|index_group_name|section_no|section_name|garment_group_no|garment_group_name|detail_desc|
+----------+------------+---------+---------------+-----------------+------------------+-----------------------+------

#Missing Values in Transactions

In [27]:
transactions_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in transactions_df.columns
]).show()

+-----+-----------+----------+-----+----------------+
|t_dat|customer_id|article_id|price|sales_channel_id|
+-----+-----------+----------+-----+----------------+
|    0|          0|         0|    0|               0|
+-----+-----------+----------+-----+----------------+



In [28]:
customers_df.groupBy("club_member_status").count().orderBy("count", ascending=False).show()

+------------------+-------+
|club_member_status|  count|
+------------------+-------+
|            ACTIVE|1272491|
|        PRE-CREATE|  92960|
|              NULL|   6062|
|         LEFT CLUB|    467|
+------------------+-------+



In [29]:
customers_df.groupBy("fashion_news_frequency").count().orderBy("count", ascending=False).show()

+----------------------+------+
|fashion_news_frequency| count|
+----------------------+------+
|                  NONE|877711|
|             Regularly|477416|
|                  NULL| 16009|
|               Monthly|   842|
|                  None|     2|
+----------------------+------+



In [30]:
customers_df.groupBy("FN").count().show()

+----+------+
|  FN| count|
+----+------+
|NULL|895050|
| 1.0|476930|
+----+------+



In [31]:
customers_df.groupBy("Active").count().show()

+------+------+
|Active| count|
+------+------+
|  NULL|907576|
|   1.0|464404|
+------+------+

